## Structured Output

Used OpenAI Structured Outputs with Pydantic schemas to extract sentiment, topic and confidence scores from customer reviews and analyze the results with pandas.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [3]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [10]:
import pandas as pd

In [4]:
from enum import Enum
from pydantic import BaseModel


class Topic(str, Enum):
    DELIVERY = "delivery"
    PRODUCT_QUALITY = "product_quality"
    CUSTOMER_SUPPORT = "customer_support"
    REFUND = "refund"
    PRICING = "pricing"
    OTHER = "other"


class Sentiment(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"


class ReviewAnalysis(BaseModel):
    sentiment: Sentiment
    topic: Topic
    confidence: float

In [5]:
review = """
The delivery was delayed by two weeks and customer support never replied.
"""

In [11]:
response = client.responses.parse(
    model="gpt-4.1-mini",
    input=f"""
Analyze the review.

Review:
{review}
""",
    text_format=ReviewAnalysis,
)

result = response.output_parsed

print(result)

sentiment=<Sentiment.NEGATIVE: 'negative'> topic=<Topic.REFUND: 'refund'> confidence=0.95


In [7]:
print(result.sentiment)
print(result.topic)
print(result.confidence)

Sentiment.NEGATIVE
Topic.DELIVERY
0.9


In [8]:
reviews = [
    "The delivery was delayed by two weeks.",
    "The product quality is excellent.",
    "Customer support was very helpful.",
    "The refund process took too long.",
]

In [12]:
results = []

for review in reviews:

    response = client.responses.parse(
        model="gpt-4.1-mini",
        input=f"Analyze the review:\n\n{review}",
        text_format=ReviewAnalysis,
    )

    result = response.output_parsed

    results.append({
        "review": review,
        "sentiment": result.sentiment,
        "topic": result.topic,
        "confidence": result.confidence,
    })

df = pd.DataFrame(results)

df

,review,sentiment,topic,confidence
0,The delivery was delayed by two weeks.,Sentiment.NEGATIVE,Topic.DELIVERY,0.95
1,The product quality is excellent.,Sentiment.POSITIVE,Topic.PRODUCT_QUALITY,0.95
2,Customer support was very helpful.,Sentiment.POSITIVE,Topic.CUSTOMER_SUPPORT,0.95
3,The refund process took too long.,Sentiment.NEGATIVE,Topic.REFUND,0.95


In [13]:
df["topic"].value_counts()

topic
Topic.DELIVERY            1
Topic.PRODUCT_QUALITY     1
Topic.CUSTOMER_SUPPORT    1
Topic.REFUND              1
Name: count, dtype: int64

In [14]:
df.groupby("topic")["confidence"].mean()

topic
Topic.CUSTOMER_SUPPORT    0.95
Topic.DELIVERY            0.95
Topic.PRODUCT_QUALITY     0.95
Topic.REFUND              0.95
Name: confidence, dtype: float64

In [15]:
df["sentiment"].value_counts()

sentiment
Sentiment.NEGATIVE    2
Sentiment.POSITIVE    2
Name: count, dtype: int64